# 01 – HDFS dataset exploration

Placeholder: explore raw HDFS logs, label distribution, and basic stats.

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

In [ ]:
BASE_DIR = Path("../machine_learning/data")

RAW_DIR = BASE_DIR / "raw"

PROCESSED_DIR = BASE_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = RAW_DIR / "HDFS.log"

print("Current working directory:", Path().resolve())
print("Log file exists:", LOG_FILE.exists())
print("Resolved path:", LOG_FILE.resolve())

In [ ]:
with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("Sample line:")
print(lines[0])

In [ ]:
pattern = re.compile(
    r"(?P<date>\d+)\s+"
    r"(?P<time>\d+)\s+"
    r"(?P<pid>\d+)\s+"
    r"(?P<level>\w+)\s+"
    r"(?P<component>[^:]+):\s+"
    r"(?P<message>.*)"
)

parsed_rows = []

for idx, line in enumerate(lines):
    match = pattern.match(line.strip())
    if match:
        row = match.groupdict()
        row["line_id"] = idx
        parsed_rows.append(row)

df = pd.DataFrame(parsed_rows)

print("Parsed rows:", len(df))
df.head()

In [ ]:
df["level"] = df["level"].str.upper()
df["component"] = df["component"].str.strip()

# Create full timestamp
df["timestamp"] = df["date"] + " " + df["time"]

df = df[[
    "line_id",
    "timestamp",
    "level",
    "component",
    "message"
]]

df.head()

In [ ]:
output_path = PROCESSED_DIR / "hdfs_structured.csv"
df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Final shape:", df.shape)

In [ ]:
LOG_PATH = LOG_FILE  # from your earlier cell

lines = LOG_PATH.read_text(errors="ignore").splitlines()
print("Total lines:", len(lines))

df = pd.DataFrame({"raw": lines})
df = df[df["raw"].str.strip().ne("")].reset_index(drop=True)
df.head(10)

In [ ]:
HDFS_PATTERN = re.compile(
    r"^(?P<date>\d{6,8})\s+(?P<time>\d{6})\s+(?P<level>[A-Z]+)\s+(?P<component>[^:]+):\s+(?P<msg>.*)$"
)

def parse_hdfs(line: str):
    m = HDFS_PATTERN.match(line)
    if not m:
        return None
    return m.groupdict()

parsed = df["raw"].apply(parse_hdfs)
df2 = pd.DataFrame([p for p in parsed if p is not None])

print("Parsed rows:", len(df2))
df2.head(10)

In [ ]:
from pathlib import Path

LOG_FILE = Path("../machine_learning/data/raw/HDFS.log").resolve()
print("LOG_FILE =", LOG_FILE)
print("exists =", LOG_FILE.exists())

In [ ]:
hdfs_pattern = re.compile(
    r"^(?P<date>\d{6})\s+(?P<time>\d{6})\s+(?P<pid>\d+)\s+"
    r"(?P<level>INFO|WARN|ERROR|DEBUG|FATAL)\s+"
    r"(?P<component>[^:]+):\s+(?P<message>.*)$"
)

rows = []
bad = 0

with open(LOG_FILE, "r", errors="ignore") as f:
    for line in f:
        line = line.rstrip("\n")
        m = hdfs_pattern.match(line)
        if not m:
            bad += 1
            continue
        rows.append(m.groupdict())

df = pd.DataFrame(rows)

print("Parsed rows:", len(df))
print("Unmatched lines:", bad)
df.head(5)

In [ ]:
import numpy as np

# --- token normalizers (HDFS-specific + general) ---
RE_BLOCK = re.compile(r"\bblk_-?\d+\b")
RE_IPPORT = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b")
RE_NUMBER = re.compile(r"\b\d+\b")
RE_HEX = re.compile(r"\b0x[0-9a-fA-F]+\b")

def to_template(msg: str) -> str:
    s = msg

    # order matters: do more specific first
    s = RE_BLOCK.sub("<BLK>", s)
    s = RE_IPPORT.sub("<IP>", s)
    s = RE_HEX.sub("<HEX>", s)

    # collapse standalone numbers last (keeps tokens like "PacketResponder" intact)
    s = RE_NUMBER.sub("<NUM>", s)

    # normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def template_id(tpl: str, n=10) -> str:
    # stable id for grouping; short prefix is fine
    h = hashlib.sha1(tpl.encode("utf-8")).hexdigest()
    return f"tpl_{h[:n]}"

# create template + template_id
df["template"] = df["message"].astype(str).apply(to_template)
df["template_id"] = df["template"].apply(template_id)

print("Unique templates:", df["template_id"].nunique())
df[["message", "template", "template_id"]].head(10)

In [ ]:
tpl_counts = (
    df.groupby(["template_id", "template"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

tpl_counts.head(20)

In [ ]:
df["template_id"].nunique(), len(df)

In [ ]:
tpl = "tpl_632f57fcfe"
df.loc[df["template_id"] == tpl, "message"].head(10).tolist()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ---- 1️⃣ Encode template_id to integers ----
le = LabelEncoder()
df["template_idx"] = le.fit_transform(df["template_id"])
num_templates = len(le.classes_)
print("Number of unique templates:", num_templates)

# ---- 2️⃣ Create sliding windows ----
WINDOW_SIZE = 20
template_sequence = df["template_idx"].values

windows = []
for i in range(len(template_sequence) - WINDOW_SIZE + 1):
    window = template_sequence[i:i+WINDOW_SIZE]
    windows.append(window)

windows = np.array(windows)
print("Total windows:", windows.shape[0])
print("Window shape:", windows.shape)

In [ ]:
# ---- 3️⃣ Convert windows to frequency vectors ----
X = np.zeros((windows.shape[0], num_templates))

for i, window in enumerate(windows):
    for tpl_idx in window:
        X[i, tpl_idx] += 1

print("Feature matrix shape:", X.shape)